<a href="https://colab.research.google.com/github/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model/blob/main/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM Research 2026 - Colab Setup

Track 1: personalize Maia-2 on a single player's games.

**Run order:** cells 1-4 every new session (Colab wipes everything on disconnect).
Cells 5+ only when you need them.

Runtime menu -> Change runtime type -> **T4 GPU** before you start.

## 1. Environment check

In [ ]:
!nvidia-smi
import sys
print(sys.version)
# maia2 supports Python 3.10-3.12. If Colab is on 3.13+, pip install will fail
# and you need a pinned env instead. Check this FIRST, not after a cryptic error.

## 2. Mount Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/acm_chess'
!mkdir -p {PROJ}/data {PROJ}/checkpoints {PROJ}/runs
print(PROJ)

## 3. Install (reruns every session)

In [ ]:
!pip install -q maia2 python-chess
import maia2, chess
print('maia2', maia2.__version__ if hasattr(maia2,'__version__') else 'installed')

## 4. Get the data

Lichess handed over data as PGN, over the API.


In [ ]:
USER = 'schwilynn'   # change per player

!curl -L -H "Accept: application/x-chess-pgn" \
  "https://lichess.org/api/games/user/{USER}?max=500&perfType=rapid&rated=true&clocks=true&evals=false" \
  -o {PROJ}/data/{USER}.pgn

!ls -lh {PROJ}/data/

## 5. Check the PGN

In [ ]:
import chess.pgn

path = f'{PROJ}/data/{USER}.pgn'
games, positions = 0, 0
with open(path) as f:
    while True:
        g = chess.pgn.read_game(f)
        if g is None:
            break
        games += 1
        positions += sum(1 for _ in g.mainline_moves())

print(f'{games} games, {positions} half-moves')
print('positions to move: ~', positions // 2)
# A few thousand of YOUR moves is the realistic scale for Track 1.

## 6. Load Maia-2 and confirm it runs

In [ ]:
from maia2 import dataset, inference, model

maia2_model = model.from_pretrained(type="rapid", device="auto")
data = dataset.load_example_test_dataset()

data, accuracy = inference.inference_batch(
    data,
    maia2_model,
    verbose=1,
    batch_size=1024,
    num_workers=0,
)
print('baseline move-matching accuracy on example set:', accuracy)

## 8. Checkpoint pattern

Colab disconnects. Save every epoch to Drive.

In [ ]:
import torch, json, time

def save_ckpt(model, optimizer, epoch, run_name, metrics=None):
    path = f'{PROJ}/checkpoints/{run_name}_epoch{epoch:02d}.pt'
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'opt': optimizer.state_dict(),
        'metrics': metrics or {},
        'ts': time.time(),
    }, path)
    with open(f'{PROJ}/runs/{run_name}.jsonl', 'a') as f:
        f.write(json.dumps({'epoch': epoch, **(metrics or {})}) + '\n')
    return path

## 9. Freeze the environment for the writeup

Run this at the end of any run whose numbers go in the report.

In [ ]:
!pip freeze > {PROJ}/runs/requirements_{USER}.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv > {PROJ}/runs/gpu.txt
print('frozen')